# Code Execution and the Files API

Alternative way of handling files, uploading files ahead of time, using file ids to reference the file itself

In [2]:
from utils.util_funcs import *

client, model = api_client_setup()

# Checking which model got loaded
print(f'Anthropic Client loaded for {client} with model: {model}')

Anthropic Client loaded for <anthropic.Anthropic object at 0x110e32c10> with model: claude-sonnet-4-5


When sending a user message, a `ContainerUploadBlock` gets attached alongside a `Textblock` object. The assistant message returns a `ServerToolUse` block to request tool use, along with a `CodeExecutionToolResultBlock`, which demonstrates the results of the code execution. This is alongside the assistant's `TextBlock` object

Claude can also execute the code inside a Docker Container to keep it internalised. Note that Claude cannot execute code directly that accesses the network without permission

In [4]:
file_metadata = upload('./supporting_materials/streaming.csv')
file_metadata

FileMetadata(id='file_011CdEGsPGpjmDFjL58xrvfX', created_at=datetime.datetime(2026, 7, 21, 0, 42, 19, 378000, tzinfo=datetime.timezone.utc), filename='streaming.csv', mime_type='text/csv', size_bytes=25733, type='file', downloadable=False, scope=None)

In [5]:
messages = []

add_user_message(
    messages,
    [
        {
            "type": "text",
            "text": """
Run a detailed analysis to determine major drivers of churn.
Your final output should include at least one detailed plot summarizing your findings.

Critical note: Every time you execute code, you're starting with a completely clean slate. 
No variables or library imports from previous executions exist. You need to redeclare/reimport all variables/libraries.
            """,
        },
        {"type": "container_upload", "file_id": file_metadata.id},
    ],
)

chat(messages, tools=[{"type": "code_execution_20250825", "name": "code_execution"}])

Message(id='msg_011CdEGtvZmyPKLANW6KCZhA', container=Container(id='container_011wRb3evKoNN8vQkD124dbc', expires_at=datetime.datetime(2026, 7, 21, 1, 43, 20, 749018, tzinfo=TzInfo(0))), content=[TextBlock(citations=None, text="I'll conduct a detailed analysis to determine the major drivers of churn in the streaming dataset. Let me start by exploring the data.", type='text'), ServerToolUseBlock(id='srvtoolu_01WWgzPjMz4PhrXgbW9f7ACp', caller=None, input={'command': 'cd $INPUT_DIR && head -20 streaming.csv && wc -l streaming.csv'}, name='bash_code_execution', type='server_tool_use'), BashCodeExecutionToolResultBlock(content=BashCodeExecutionResultBlock(content=[], return_code=0, stderr='', stdout='UserID,SubscriptionTier,TotalViewingHoursLastMonth,TopGenre,BingeWatchingSessionsLastMonth,NumberOfUniqueTitlesWatchedLastMonth,AverageSessionDurationMinutes,CustomerServiceInteractionsLastYear,MonthlyCost,Churned\nUSER_00001,Basic,47.9,Comedy,5,15,32.6,3,7.99,0\nUSER_00002,Premium,41.4,Drama,5,9

In [ ]:
download_file('')